# Lesson 3A: DBSCAN - Theory

## Introduction

You are analyzing GPS data from taxi rides in a city. You want to find popular pickup locations, but:
- **K-Means fails**: Pickup areas have arbitrary shapes (not spherical)
- **Hierarchical fails**: Noise points (isolated rides) should not form clusters
- **You do not know k**: Number of hotspots varies by time of day

This is exactly what **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) solves!

**Real-world applications**:
- **GPS data**: Find hotspots in location data
- **Anomaly detection**: Identify outliers as noise points
- **Image segmentation**: Find arbitrary-shaped objects
- **Astronomy**: Discover star clusters in sky surveys

In this lesson:
1. Understand density-based clustering concepts
2. Learn about core points, border points, and noise
3. Implement DBSCAN from scratch
4. Apply to non-spherical cluster data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, make_circles
from typing import List, Set
from numpy.typing import NDArray

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries loaded!")

## DBSCAN Concepts

### Key Parameters

- **ε (epsilon)**: Neighborhood radius
- **MinPts**: Minimum points to form dense region

### Point Types

**1. Core Point**: Has ≥ MinPts neighbors within ε

**2. Border Point**: Within ε of core point but has < MinPts neighbors  

**3. Noise Point**: Not core or border

### Algorithm

1. For each unvisited point p:
   - If p has < MinPts neighbors: mark as noise
   - If p has ≥ MinPts neighbors: start new cluster, expand
2. Expansion: Add all density-reachable points

In [ ]:
class DBSCAN:
    """DBSCAN clustering from scratch."""
    
    def __init__(self, eps=0.5, min_pts=5):
        self.eps = eps
        self.min_pts = min_pts
        self.labels = None
        
    def fit(self, X: NDArray):
        n = len(X)
        self.labels = np.full(n, -1)  # -1 = noise
        cluster_id = 0
        
        for i in range(n):
            if self.labels[i] != -1:
                continue
                
            neighbors = self._get_neighbors(X, i)
            
            if len(neighbors) < self.min_pts:
                self.labels[i] = -1  # Noise
            else:
                self._expand_cluster(X, i, neighbors, cluster_id)
                cluster_id += 1
                
        return self
        
    def _get_neighbors(self, X: NDArray, point_idx: int) -> List[int]:
        neighbors = []
        for i in range(len(X)):
            if np.linalg.norm(X[point_idx] - X[i]) < self.eps:
                neighbors.append(i)
        return neighbors
        
    def _expand_cluster(self, X: NDArray, point_idx: int, 
                       neighbors: List[int], cluster_id: int):
        self.labels[point_idx] = cluster_id
        i = 0
        while i < len(neighbors):
            neighbor = neighbors[i]
            if self.labels[neighbor] == -1:
                self.labels[neighbor] = cluster_id
            elif self.labels[neighbor] == -1:
                self.labels[neighbor] = cluster_id
                new_neighbors = self._get_neighbors(X, neighbor)
                if len(new_neighbors) >= self.min_pts:
                    neighbors.extend(new_neighbors)
            i += 1

print("✅ DBSCAN implemented!")

In [ ]:
# Test on non-spherical data
X, _ = make_moons(n_samples=200, noise=0.05, random_state=42)

dbscan = DBSCAN(eps=0.3, min_pts=5)
dbscan.fit(X)

plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], c=dbscan.labels, cmap='viridis', edgecolors='k')
plt.title('DBSCAN on Moon Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.colorbar(label='Cluster (-1 = noise)')
plt.show()

print(f"✅ Found {len(set(dbscan.labels)) - (1 if -1 in dbscan.labels else 0)} clusters!")
print(f"Noise points: {sum(dbscan.labels == -1)}")

## Conclusion

DBSCAN advantages:
- ✅ Finds arbitrary-shaped clusters
- ✅ Identifies noise/outliers
- ✅ No need to specify number of clusters
- ✅ Works well with density variations

Use DBSCAN when clusters have non-spherical shapes or outliers are present!